# Station Boardings / Alightings

**Purpose:** Summarize station boardings and alightings for the Mid-Coast project station, also summarize the station-to-station table.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [2]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df = pd.read_csv(interim_dir + "Interim_2023_OnBoardSurvey_Dataset_1.csv")

C:\Users\USXH723910\AppData\Local\Temp\ipykernel_2064\1585012778.py:10: DtypeWarning: Columns (17,32,35,36,37,39,40,57,61,62,64,65,79,81,95,97,115,117,119,121,194,201,202,203,205,207) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(interim_dir + "Interim_2023_OnBoardSurvey_Dataset_1.csv")


### Define Boardings and Alightings Station in PA format

In [3]:
# Creating station boardings and alightings in PA format
df['STOP_Prod'] = np.where(df['PA_FLAG'] == 0, df['STOP_OFF [ADDR]'], df['STOP_ON [ADDR]'])
df['STOP_Attr'] = np.where(df['PA_FLAG'] == 0, df['STOP_ON [ADDR]'], df['STOP_OFF [ADDR]'])


In [4]:
# Creating the survey route direction in PA format
df['Direction_reverse'] = None
df['Direction_PA'] = None
df['Direction_reverse'] = np.where(df['DIRECTION']==0, 1, np.where(df['DIRECTION']==1, 0, None))
df['Direction_PA'] = np.where(df['PA_FLAG'] == 0, df['Direction_reverse'], df['DIRECTION'])

Station Boardings and Alightings on Mid-Coast Project Stations

In [5]:
project_trips = df[df['project_trip_update']=='yes'].copy()

In [12]:
midcoast_stns = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

## Generate Station-to-Station Table

In [20]:
def creating_station_to_station_table(df,index_name, column_name,value):
    pivot = pd.pivot_table(
        df,
        index=index_name,
        columns=column_name,
        values=value,
        aggfunc='sum',
        fill_value=0,
    )

    # Filter pivot to selected stations
    filtered_pivot = pivot.loc[pivot.index.isin(midcoast_stns), midcoast_stns]

    # Add 'Total' column- row sum
    totals_by_column = df.groupby(index_name)[value].sum()
    filtered_pivot['Total'] = totals_by_column.reindex(filtered_pivot.index).fillna(0)

    # Add 'Total' row- column sum
    totals_by_row = df.groupby(column_name)[value].sum()
    total_row = totals_by_row.reindex(filtered_pivot.columns, fill_value=0)
    filtered_pivot.loc['Total'] = total_row


    filtered_pivot.loc['Total', 'Total'] = df[value].sum()


    filtered_pivot['Other'] = filtered_pivot['Total'] - filtered_pivot.drop(columns=['Total']).sum(axis=1)
    filtered_pivot.loc['Other'] = filtered_pivot.loc['Total'] - filtered_pivot.drop(index='Total').sum(axis=0)

    order_list = midcoast_stns + ['Other', 'Total']
    filtered_pivot = filtered_pivot.loc[order_list, order_list]
    filtered_pivot = filtered_pivot.reset_index().round(1)
    return filtered_pivot

### Table 7 in Report
Table 7 Daily Station-to-Station Transit Linked Trips (PA format)

In [21]:
df_station_PA = creating_station_to_station_table(project_trips,'STOP_Prod', 'STOP_Attr', 'REWEIGHTED_LINKED')
df_station_PA.to_csv(output_dir + 'Table_7_Station_to_Station_PA.csv', index=False)


Notes: Table 4 (Daily Station Boardings and Alightings on Mid-Coast Project Stations) is calculated based on the Table 7

For example, Balboa Avenue Southbound On should be the sum of Balboa Ave – Clairmont Dr, Balboa Ave – Tecolote Rd, and Balboa Ave – Other in the station-to-station table. So, 34 + 33 + 1256 = 1323.

For Nobel Drive Northbound Off, it should include Balboa Ave – Nobel Dr, Clairmont Dr–Nobel Dr, Tecolote Rd – Nobel Dr, and Other – Nobel Dr. Therefore, 189 + 62 + 0 + 693 = 943.

### Table 21 in Report
Table 21 Daily Station-to-Station Transit Linked Trips (OD format)

In [22]:
df_station_OD_unbalanced = creating_station_to_station_table(project_trips,'STOP_ON [ADDR]', 'STOP_OFF [ADDR]', 'REWEIGHTED_LINKED')
df_station_OD_unbalanced.to_csv(output_dir + 'Table_21_Station_to_Station_OD_unbalanced.csv', index=False)

In [23]:
station_columns = df_station_OD_unbalanced.columns.drop(['STOP_ON [ADDR]', 'Total'])
od_matrix = df_station_OD_unbalanced[station_columns].copy()
od_matrix.index = df_station_OD_unbalanced['STOP_ON [ADDR]']

# Symmetrize the matrix- average
df_station_OD_balanced = (od_matrix + od_matrix.T) / 2

# Recompute totals for balanced matrix
df_station_OD_balanced['Total'] = df_station_OD_balanced.sum(axis=1)
df_station_OD_balanced.loc['Total'] = df_station_OD_balanced.sum()

order_list = midcoast_stns + ['Other', 'Total']
df_station_OD_balanced = df_station_OD_balanced.reset_index().rename(columns={'index': 'STOP_ON [ADDR]'})
df_station_OD_balanced = df_station_OD_balanced.set_index('STOP_ON [ADDR]')
df_station_OD_balanced = df_station_OD_balanced.loc[order_list]


df_station_OD_balanced = df_station_OD_balanced[order_list]
df_station_OD_balanced = df_station_OD_balanced.reset_index()  
df_station_OD_balanced.to_csv(output_dir + 'Table_21_Station_to_Station_OD_balanced.csv', index=False)

### Table 8 in Report
Daily Mid-Coast Station Mode of Access and Egress, 2023 Transit On-Board Survey

In [24]:
# PA format
df_stop_prod = df[df['stop_prod_project'].isin(midcoast_stns)][['stop_prod_project','direction_pa_project','access_w_xfer_pa_project','REWEIGHTED_LINKED']]
df_ProjectBoardings = df_stop_prod.groupby(['direction_pa_project','stop_prod_project','access_w_xfer_pa_project']).sum('REWEIGHTED_LINKED')

In [25]:
df_ProjectBoardings = df_ProjectBoardings.reset_index()
# df_ProjectBoardings['Direction_PA'] = df_ProjectBoardings['Direction_PA'].map({0: 'Northbound', 1: 'Southbound'})

# Pivot the table: make ACCESS_MODE_wXFER_PA columns, with REWEIGHTED_LINKED values
df_ProjectBoardings_pivot = df_ProjectBoardings.pivot_table(
    index=['direction_pa_project', 'stop_prod_project'], 
    columns='access_w_xfer_pa_project', 
    values='REWEIGHTED_LINKED', 
    aggfunc='sum',
    fill_value=0
)

# Reorder columns as desired
df_ProjectBoardings_pivot = df_ProjectBoardings_pivot[['Walk', 'KNR', 'PNR', 'Xfer']]

df_ProjectBoardings_pivot = df_ProjectBoardings_pivot.reset_index().round(1)
# df_ProjectBoardings_pivot.round(1).to_csv(output_dir + 'Table_8_Station_Mode_of_Access.csv',index=False)

In [26]:
df_stop_attr = df[df['stop_attr_project'].isin(midcoast_stns)][['stop_attr_project','direction_pa_project','egress_w_xfer_pa_project','REWEIGHTED_LINKED']]
df_ProjectAlightings = df_stop_attr.groupby(['direction_pa_project','stop_attr_project','egress_w_xfer_pa_project']).sum('REWEIGHTED_LINKED')
# df_ProjectAlightings

In [27]:
df_ProjectAlightings = df_ProjectAlightings.reset_index()
# df_ProjectAlightings['Direction_PA'] = df_ProjectAlightings['Direction_PA'].map({0: 'Northbound', 1: 'Southbound'})

# Pivot the table: make ACCESS_MODE_wXFER_PA columns, with REWEIGHTED_LINKED values
df_ProjectAlightings_pivot = df_ProjectAlightings.pivot_table(
    index=['direction_pa_project', 'stop_attr_project'], 
    columns='egress_w_xfer_pa_project', 
    values='REWEIGHTED_LINKED', 
    aggfunc='sum',
    fill_value=0
)

# Reorder columns as desired
df_ProjectAlightings_pivot = df_ProjectAlightings_pivot[['Walk', 'KNR', 'PNR', 'Xfer']]

df_ProjectAlightings_pivot = df_ProjectAlightings_pivot.reset_index().round(1)
# df_ProjectAlightings_pivot.round(1).to_csv(output_dir + 'Table_8_Station_Mode_of_Egress.csv',index=False)

### Appendix F in Report

Blue Line Trolley Station Boardings and Alightings (OD format)

In [28]:
Blue_Line_Stns = ['UTC Station',
        'Executive Drive Station',
        'UCSD Health La Jolla Station',
        'UCSD Central Campus Station',
        'VA Medical Center Station',
        'Nobel Drive Station',
        'Balboa Avenue Station',
        'Clairemont Drive Station',
        'Tecolote Road Station',
        'Old Town Station',
        'Washington Street Station',
        'Middletown Station',
        'County Center/Little Italy Station',
        'Santa Fe Depot',
        'America Plaza Station',
        'Civic Center Station',
        'Fifth Avenue Station',
        'City College Station',
        'Park & Market Station',
        '12th & Imperial Station',
        'Barrio Logan Station',
        'Harborside Station',
        'Pacific Fleet Station',
        '8th Street Station',
        '24th Street Station',
        'E Street Station',
        'H Street Station',
        'Palomar Street Station',
        'Palm Avenue Station',
        'Iris Avenue Station',
        'Beyer Blvd Station',
        'San Ysidro Station']

In [29]:
# Filter out the Blue Line survey record
df_blue = df[df['ROUTE_NUMBER']=='Blue']

In [30]:
# OD Format
df_blue_on = df_blue[df_blue['STOP_ON [ADDR]'].isin(Blue_Line_Stns)][['STOP_ON [ADDR]','DIRECTION','REWEIGHTED_UNLINKED']]
df_BlueBoardings = df_blue_on.groupby(['DIRECTION','STOP_ON [ADDR]']).sum('REWEIGHTED_UNLINKED').reset_index()

df_BlueBoardings['DIRECTION'] = df_BlueBoardings['DIRECTION'].map({0: 'Northbound', 1: 'Southbound'})
df_BlueBoardings.to_csv(output_dir + 'Table_Appendix_F_Blue_Line_Station_Boardings.csv',index=False)
df_BlueBoardings

,DIRECTION,STOP_ON [ADDR],REWEIGHTED_UNLINKED
0,Northbound,12th & Imperial Station,2782.482337
1,Northbound,24th Street Station,936.919860
2,Northbound,8th Street Station,851.393625
3,Northbound,America Plaza Station,937.648239
4,Northbound,Balboa Avenue Station,745.435994
...,...,...,...
58,Southbound,UCSD Central Campus Station,3041.071566
59,Southbound,UCSD Health La Jolla Station,1098.794495
60,Southbound,UTC Station,2723.835196
61,Southbound,VA Medical Center Station,1472.427474


In [31]:
df_blue_off = df_blue[df_blue['STOP_OFF [ADDR]'].isin(Blue_Line_Stns)][['STOP_OFF [ADDR]','DIRECTION','REWEIGHTED_UNLINKED']]
df_BlueAlightings = df_blue_off.groupby(['DIRECTION','STOP_OFF [ADDR]']).sum('REWEIGHTED_UNLINKED').reset_index()
df_BlueAlightings['DIRECTION'] = df_BlueAlightings['DIRECTION'].map({0: 'Northbound', 1: 'Southbound'})
df_BlueAlightings.to_csv(output_dir + 'Table_Appendix_F_Blue_Line_Station_Alightings.csv',index=False)
df_BlueAlightings

,DIRECTION,STOP_OFF [ADDR],REWEIGHTED_UNLINKED
0,Northbound,12th & Imperial Station,3831.319019
1,Northbound,24th Street Station,1162.289667
2,Northbound,8th Street Station,777.511244
3,Northbound,America Plaza Station,3912.013114
4,Northbound,Balboa Avenue Station,1199.113221
...,...,...,...
56,Southbound,Tecolote Road Station,461.301136
57,Southbound,UCSD Central Campus Station,1973.529879
58,Southbound,UCSD Health La Jolla Station,165.924678
59,Southbound,VA Medical Center Station,95.707381


In [32]:
df.to_csv(interim_dir + 'Interim_2023_OnBoardSurvey_Dataset_2.csv',index=False)